In [ ]:
import leafmap
file_path = "./Unprocessed/clipped_LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF"
m = leafmap.Map(center=getLongitudeLatitudeOfTif(file_path), zoom=6)
m.add_raster(file_path, layer_name="Raster Layer")
m

In [1]:
import os
import shutil

def clear_folder(folder_path):
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)
                print(f"Deleted file: {file_path}")
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
                print(f"Deleted folder: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")

# Call the function
clear_folder('./RawClippedRasters')


Deleted folder: ./RawClippedRasters\Albedo
Deleted folder: ./RawClippedRasters\DEM
Deleted folder: ./RawClippedRasters\Land_Cover
Deleted folder: ./RawClippedRasters\LST
Deleted folder: ./RawClippedRasters\NDVI
Deleted folder: ./RawClippedRasters\NDWI


In [ ]:
# Define search payload (if filtering is needed, adjust accordingly)
dataset_search_payload = {}

# Send request to dataset search endpoint
datasets = sendRequest(serviceUrl + "dataset-search", dataset_search_payload, apiKey)

# Print the names of available datasets
for ds in datasets:
    # print(ds.keys())
    print(f"Dataset Name: {ds['datasetCategoryName']} + -> {ds['datasetAlias']}")

In [13]:
from pynlcd import get_land_cover
from osgeo import ogr
import os

# Define file paths
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_San_Antonio_TX.shp"
shapefile_path = shapefile_folder + shapefile

# Load shapefile using GDAL/OGR
driver = ogr.GetDriverByName('ESRI Shapefile')
dataSource = driver.Open(shapefile_path, 0)  # 0 means read-only, 1 means writable
if dataSource is None:
    raise FileNotFoundError(f"Shapefile not found at {shapefile_path}")

layer = dataSource.GetLayer()
feature = layer.GetNextFeature()
if feature is None:
    raise ValueError("No features found in shapefile.")

# Extract ROI geometry
roi_geom = feature.GetGeometryRef()

# Define extent from geometry bounds
min_x, max_x, min_y, max_y = roi_geom.GetEnvelope()
extent = (min_x, max_x, min_y, max_y)

# Download the land cover data
output_path = './Unprocessed/'
get_land_cover(roi_geom, extent, year=2001, spatial_resolution=0.0003, output_path=output_path)
print(f"Land cover data saved to {output_path}")

Getting the image...
Image saved to ./Unprocessed/NLCD_2001_Land_Cover.tif
Land cover data saved to ./Unprocessed/


In [ ]:
centroid = aoi_geodf.geometry.centroid.iloc[0]
m = folium.Map(
    location=[centroid.y, centroid.x], 
    zoom_start=9, tiles="openstreetmap", width="100%", height="100%", attributionControl=0
)
# Cell 4: Add Polygon to Map
folium.GeoJson(aoi_geodf).add_to(m)
m

In [ ]:
# from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd

# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_Pahrump_NV.shp"
city = shapefile.replace('Polygon_', '').replace('.shp', '')
aoi_geodf = gpd.read_file(shapefile_folder + shapefile)
aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
if aoi_geodf.empty:
    sys.exit("Error: Shapefile contains no data.")
print("Shapefile loaded successfully.")
import warnings

# Reset warning filters and display all warnings
warnings.resetwarnings()  # Resets any previously set filters
warnings.filterwarnings("default")  # Displays all warnings, including once-per-session ones

import rioxarray
import rasterio
import os

def clipUnprocessedRasters(tifs, boundPolygon):
    # Reproject the bounding polygon to EPSG:4326
    if boundPolygon.crs != "EPSG:4326":
        print("Reprojecting bounding polygon to EPSG:4326...")
        boundPolygon = boundPolygon.to_crs("EPSG:4326")
    
    goodCoordinates = []
    for tif in tifs:
        tif_path = os.path.join(unprocessed_dir, tif)

        # Open the raster using rioxarray
        raster = rioxarray.open_rasterio(tif_path)
        print(f"Opened raster: {tif}, Original CRS: {raster.rio.crs}")

        # Reproject the raster to EPSG:4326
        raster_reprojected = raster.rio.reproject("EPSG:4326")
        print(f"Raster reprojected to EPSG:4326 for clipping.")

        # Extract colormap from the original raster
        colormap = None
        with rasterio.open(tif_path) as src:
            if src.colorinterp[0] == rasterio.enums.ColorInterp.palette:
                colormap = src.colormap(1)  # Assuming band 1 has the colormap

        # Clip the raster using the bounding polygon
        clipped = raster_reprojected.rio.clip(boundPolygon.geometry, boundPolygon.crs, drop=True)
        print(f"Clipping completed for raster: {tif}")

        # Save the output raster, preserving metadata
        clipped_file_path = os.path.join(unprocessed_dir, f"Clipped_{tif}")
        clipped.rio.to_raster(clipped_file_path)
        print(f"Clipped raster saved at: {clipped_file_path}")

        # Reapply colormap to the saved raster if it exists
        if colormap:
            with rasterio.open(clipped_file_path, "r+") as dest:
                dest.write_colormap(1, colormap)
            print(f"Colormap reapplied to raster: {clipped_file_path}")

        # Add the coordinate info from filename to the list
        goodCoordinates.append(tif.split('_')[2])
        print(f"Processed and saved TIF: {clipped_file_path}")

    return goodCoordinates



goodCoordinates = clipUnprocessedRasters(['LC08_L2SP_040035_20130810_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130810_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130810_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130810_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20131216_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130810_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130810_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130420_20200912_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_SR_B2.TIF', 'LC08_L2SP_040035_20130709_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20131130_20200912_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130319_20200913_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_SR_B4.TIF', 'LC08_L2SP_040035_20130623_20200912_02_T1_SR_B6.TIF', 'LC08_L2SP_040035_20130607_20200912_02_T1_SR_B5.TIF', 'LC08_L2SP_040035_20130725_20200912_02_T1_ST_B10.TIF', 'LC08_L2SP_040035_20130522_20200913_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20130927_20200913_02_T1_SR_B3.TIF', 'LC08_L2SP_040035_20131013_20200912_02_T1_SR_B6.TIF'], aoi_geodf)